**Large Dataset Handling Practices** 

```
Handling Data:

usecols
dtype
chunksize
nrows
category
memory_usage()
```
Export Data: 

| متد            | فرمت    | کاربرد                          |
| -------------- | ------- | ------------------------------- |
| `to_csv()`     | CSV     | رایج‌ترین و ساده‌ترین           |
| `to_excel()`   | Excel   | کار با فایل‌های Excel           |
| `to_json()`    | JSON    | API / Web / Data Exchange       |
| `to_parquet()` | Parquet | دیتاست‌های بزرگ و Data/ML       |
| `to_pickle()`  | Pickle  | ذخیره‌ی خود DataFrame در Python |

---

In [38]:
import os

import numpy as np
import pandas as pd

In [39]:
file_path = "../data/large_predictions.csv"

if not os.path.exists(file_path):
    np.random.seed(42)

    n = 100_000

    large_df = pd.DataFrame(
        {
            "image_id": [f"img_{i:07d}" for i in range(n)],
            "model": np.random.choice(["CNN", "ResNet", "ViT", "EfficientNet"], size=n),
            "dataset": np.random.choice(["CIFAR10", "CIFAR100", "ImageNet"], size=n),
            "label": np.random.randint(0, 100, size=n),
            "prediction": np.random.randint(0, 100, size=n),
            "confidence": np.random.uniform(0.5, 1.0, size=n),
            "inference_time_ms": np.random.uniform(5, 150, size=n),
            "batch_size": np.random.choice([16, 32, 64, 128], size=n),
            "gpu": np.random.choice(["RTX_3090", "RTX_4090", "A100", "H100"], size=n),
            "experiment_id": np.random.randint(1, 1000, size=n),
            "unused_feature_1": np.random.randn(n),
            "unused_feature_2": np.random.randn(n),
        }
    )

    large_df.to_csv(file_path, index=False)

    del large_df

    print("Dataset created.")
else:
    print("Dataset already exists. Nothing was created.")

Dataset created.


In [40]:
# Part 1 — بررسی اندازه فایل

size_mb = os.path.getsize(file_path) / (1024**2)

print(f"File size: {size_mb:.2f} MB")

File size: 11.93 MB


In [41]:
# Part 2 — خواندن فقط چند ستون

df = pd.read_csv(file_path, usecols=["image_id", "model", "confidence", "prediction"])

df.head()

,image_id,model,prediction,confidence
0,img_0000000,ViT,3,0.597601
1,img_0000001,EfficientNet,46,0.724214
2,img_0000002,CNN,0,0.635269
3,img_0000003,ViT,40,0.521845
4,img_0000004,ViT,21,0.886361


In [42]:
# Part 3 — بررسی Memory
df.info(memory_usage="deep")

# یا

df.memory_usage(deep=True)

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   image_id    100000 non-null  str    
 1   model       100000 non-null  str    
 2   prediction  100000 non-null  int64  
 3   confidence  100000 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 12.5 MB


Index             132
image_id      6000000
model         5500021
prediction     800000
confidence     800000
dtype: int64

In [43]:
# Part 4 — تبدیل model به Category
df["model"] = df["model"].astype("category")

df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   image_id    100000 non-null  str     
 1   model       100000 non-null  category
 2   prediction  100000 non-null  int64   
 3   confidence  100000 non-null  float64 
dtypes: category(1), float64(1), int64(1), str(1)
memory usage: 7.3 MB


In [44]:
# Part 5 — استفاده از chunksize

chunks = pd.read_csv(
    file_path, usecols=["model", "confidence", "prediction"], chunksize=100_000
)

In [45]:
# Part 6 — پردازش Chunkها

model_counts = {}

for chunk in chunks:
    counts = chunk["model"].value_counts()

    for model, count in counts.items():
        model_counts[model] = model_counts.get(model, 0) + count

model_counts

{'ResNet': 25091, 'CNN': 25049, 'EfficientNet': 24972, 'ViT': 24888}

In [46]:
# Part 7 — محاسبه میانگین Confidence

chunks = pd.read_csv(file_path, usecols=["confidence"], chunksize=100_000)

total_confidence = 0
total_rows = 0

for chunk in chunks:
    total_confidence += chunk["confidence"].sum()
    total_rows += len(chunk)

mean_confidence = total_confidence / total_rows

mean_confidence

np.float64(0.7500747510528717)